# Chapter 12: Augmenting Investment Decisions — LLM-Guided Portfolio Allocation
**Financial AI in Practice** | Manning Publications

This notebook covers:
- Integrating quantitative momentum signals with LLM-based macro interpretation
- Designing system prompts that provide bounded discretion within explicit constraints
- Building a historical simulation that respects information timing
- Understanding the gap between simulation and live trading
- Positioning human judgment as the final decision layer

## 12.2 Setting Up the Simulation Infrastructure
### Listing 12.1 — Configuration and portfolio universe

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from fredapi import Fred
import yfinance as yf
from openai import OpenAI
import json
from dotenv import load_dotenv, find_dotenv
import os

# API Configuration: keys come from the repo-root .env (appendix A); copy .env.example to .env
load_dotenv(find_dotenv(usecwd=True))
for var in ("FRED_API_KEY", "OPENAI_API_KEY"):
    if not os.getenv(var):
        raise RuntimeError(f"{var} is not set: copy .env.example to .env in the repo root and add your key.")
fred = Fred(api_key=os.getenv("FRED_API_KEY"))        #A
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  #B

# Portfolio Universe
ETF_LIST = ['SPY', 'QQQ', 'VIXY', 'TLT', 'WEAT']  #C
INITIAL_WEIGHT = 20                                  #D

# Simulation Parameters
REBALANCE_DATES = ['2025-04-01', '2025-05-01', '2025-06-01',
                   '2025-07-01', '2025-08-01', '2025-09-01',
                   '2025-10-01']                     #E

#A FRED API client for macroeconomic data access
#B OpenAI client for LLM recommendations
#C Five-ETF universe including intentionally challenging instruments (VIXY, WEAT)
#D Equal weight starting point: 20% each across five ETFs
#E Monthly rebalancing dates defining our simulation periods

### Listing 12.2 — Collecting macroeconomic and price data

In [ ]:
DATA_START = '2024-01-01'
DATA_END = '2025-12-31'

FRED_CODES = ['FEDFUNDS', 'VIXCLS', 'DGS10', 'DGS2',
              'UNRATE', 'CPIAUCSL', 'UMCSENT', 'M2SL']  #A

macro_data = {}
for code in FRED_CODES:
    try:
        series = fred.get_series(code, DATA_START, DATA_END)
        macro_data[code] = series.dropna()  #B
    except Exception as e:
        print(f"Warning: Could not fetch {code}: {e}")

# ETF prices via yfinance
etf_raw = yf.download(ETF_LIST, start=DATA_START, end=DATA_END, auto_adjust=True)  #C
etf_prices = etf_raw['Close'].dropna()

#A Eight key indicators: rates, volatility, yields, employment, inflation, sentiment, money supply
#B Store each series separately to apply indicator-specific publication lags later
#C auto_adjust=True ensures prices reflect splits and dividends

## 12.3 Designing the System Prompt
### Listing 12.3 — System prompt with explicit constraints

In [ ]:
SYSTEM_PROMPT = """You are a macro strategist analyzing portfolio allocation.
You manage a 5-ETF portfolio and recommend allocations based on macroeconomic data.

YOUR ETF UNIVERSE:
- SPY: S&P 500 index. US large-cap equities.
- QQQ: Nasdaq 100. Tech-heavy, growth-oriented.
- VIXY: VIX short-term futures. Short-term volatility hedge only.
- TLT: 20+ year Treasury bonds. Rate-sensitive, defensive.
- WEAT: Wheat commodity ETF. Agricultural/inflation exposure.

HARD CONSTRAINTS:
- Total allocation must equal exactly 100%
- Each ETF: minimum 0%, maximum 50%
- Maximum position change per asset: 15% per month
- VIXY is a tactical hedge; flag if allocation exceeds 10%

OUTPUT: JSON only, no additional text."""

# Callouts A-C annotate the three constraint lines above (max 50%, 15% per month, VIXY 10% flag).
# They are kept outside the string so they are not sent to the LLM as part of the prompt.
#A Concentration limit prevents extreme single-asset bets
#B Turnover constraint forces gradualism and reduces transaction costs
#C Domain knowledge encoded as flagging rule for human review

## 12.4 Quantitative Signal Generation
### Listing 12.4 — Momentum and volatility signals

In [ ]:
def generate_quant_signals(prices, target_date):
    """Generate momentum and volatility signals for each ETF."""
    available = prices[prices.index <= target_date]  #A
    if len(available) < 60:
        return {}
    signals = {}
    for etf in ETF_LIST:
        p = available[etf]
        mom_20d = (p.iloc[-1] / p.iloc[-20] - 1) * 100  #B
        mom_60d = (p.iloc[-1] / p.iloc[-60] - 1) * 100  #C
        vol = p.pct_change().iloc[-20:].std() * np.sqrt(252) * 100  #D
        signals[etf] = {
            'momentum_20d': round(mom_20d, 2),
            'momentum_60d': round(mom_60d, 2),
            'volatility': round(vol, 2)
        }
    return signals

#A Only use data available as of the decision date — no lookahead
#B Short-term momentum captures recent trend direction
#C Longer-term momentum captures sustained moves vs mean reversion
#D Annualized volatility from 20-day rolling window

### Listing 12.5 — Point-in-time macro extraction

In [ ]:
def get_available_macro(target_date, macro_data):
    """Extract macro data respecting publication lags."""
    target_dt = datetime.strptime(target_date, '%Y-%m-%d')
    lags = {
        'VIXCLS': 1, 'DGS10': 1, 'DGS2': 1, 'FEDFUNDS': 1,  #A
        'M2SL': 45, 'CPIAUCSL': 14, 'UNRATE': 7, 'UMCSENT': 14  #B
    }
    result = {}
    for code, lag in lags.items():
        if code in macro_data:
            cutoff = target_dt - timedelta(days=lag)
            filtered = macro_data[code][macro_data[code].index <= cutoff]  #C
            if len(filtered) > 0:
                result[code] = round(filtered.iloc[-1], 2)
    if 'DGS10' in result and 'DGS2' in result:
        result['YIELD_CURVE'] = round(result['DGS10'] - result['DGS2'], 2)  #D
    return result

#A Market data (VIX, yields) publishes daily with minimal lag
#B Economic releases have longer lags: M2 takes 45 days, CPI ~14 days
#C Filter to only include data published before the cutoff date
#D Compute yield curve spread as derived indicator

## 12.5 The LLM Portfolio Advisor
### Listing 12.6 — Constructing the user prompt

In [ ]:
def build_user_prompt(macro_data, quant_signals, current_weights):
    """Construct the user prompt with all available information."""

    macro_text = "\n".join([f"  {k}: {v}" for k, v in macro_data.items()])  #A

    quant_lines = []
    for etf, sig in quant_signals.items():
        line = f"  {etf}: 20d={sig['momentum_20d']:+.1f}%, "
        line += f"60d={sig['momentum_60d']:+.1f}%, vol={sig['volatility']:.1f}%"
        quant_lines.append(line)
    quant_text = "\n".join(quant_lines)  #B

    portfolio_text = ", ".join(
        [f"{etf}:{w}%" for etf, w in current_weights.items()]
    )  #C

    return f"""MACROECONOMIC INDICATORS:
{macro_text}

QUANTITATIVE SIGNALS:
{quant_text}

CURRENT PORTFOLIO: {portfolio_text}

Recommend target allocations.

OUTPUT (JSON only):
{{
    "reasoning": "One sentence key insight",
    "target_weights": {{"SPY": X, "QQQ": X, "VIXY": X, "TLT": X, "WEAT": X}},
    "conviction": 1-10,
    "flags": ["any constraint violations or concerns"]
}}"""

#A Format macro indicators as readable key-value pairs
#B Format quant signals showing momentum direction with +/- signs
#C Include current weights so LLM can respect turnover constraints

### Listing 12.7 — LLM API call

In [ ]:
def get_llm_recommendation(macro_data, quant_signals, current_weights):
    """Get portfolio recommendation from LLM."""
    user_prompt = build_user_prompt(macro_data, quant_signals, current_weights)
    response = client.chat.completions.create(
        model="gpt-4o-mini",  #A
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3  #B
    )
    return response.choices[0].message.content

#A gpt-4o-mini balances capability with cost for repeated simulation calls
#B Low temperature (0.3) reduces output variability while allowing reasoning flexibility

## 12.6 Running the Historical Simulation
### Listing 12.8 — Utility functions for simulation

In [ ]:
def parse_json_response(text):
    """Extract JSON from an LLM response, tolerant of surrounding prose or code fences."""
    import re
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE)  #A
    start, end = cleaned.find('{'), cleaned.rfind('}') + 1
    if start != -1 and end > start:
        try:
            return json.loads(cleaned[start:end])
        except json.JSONDecodeError as e:
            print(f"[parse] JSON decode failed: {e}. Snippet: {cleaned[start:start+80]!r}")
    return None

def normalize_weights(weights):
    """Ensure weights sum to 100%."""
    total = sum(weights.values())
    if total == 0:
        return {etf: 20 for etf in ETF_LIST}  #B
    return {k: round(v * 100 / total, 1) for k, v in weights.items()}

def calculate_period_return(weights, start_prices, end_prices):
    """Calculate portfolio return for a period."""
    total_return = 0
    for etf, weight in weights.items():
        if weight > 0:
            etf_return = (end_prices[etf] / start_prices[etf] - 1)
            total_return += (weight / 100) * etf_return  #C
    return total_return * 100

#A Extract JSON even if LLM includes surrounding text
#B Fallback to equal weights if parsing fails completely
#C Weight-adjusted return aggregation across all positions

### Listing 12.9 — Main simulation loop

In [ ]:
bh_weights = {etf: INITIAL_WEIGHT for etf in ETF_LIST}  #A
hybrid_weights = {etf: INITIAL_WEIGHT for etf in ETF_LIST}
simulation_log = []

for i in range(len(REBALANCE_DATES) - 1):
    decision_date = REBALANCE_DATES[i]
    next_date = REBALANCE_DATES[i + 1]

    # Get available data as of decision date
    macro = get_available_macro(decision_date, macro_data)
    quant = generate_quant_signals(etf_prices, decision_date)

    # Get LLM recommendation
    response = get_llm_recommendation(macro, quant, hybrid_weights)
    parsed = parse_json_response(response)

    if parsed and 'target_weights' in parsed:
        new_weights = normalize_weights(parsed['target_weights'])
        reasoning = parsed.get('reasoning', '')
        conviction = parsed.get('conviction', 5)
        flags = parsed.get('flags', [])
    else:
        new_weights = hybrid_weights.copy()  #B
        reasoning = 'Parse error - maintaining current weights'
        conviction = 0
        flags = ['parse_failure']

    # Calculate returns using START-of-period weights
    start_p = etf_prices.loc[etf_prices.index <= decision_date].iloc[-1]
    end_p = etf_prices.loc[etf_prices.index <= next_date].iloc[-1]
    bh_ret = calculate_period_return(bh_weights, start_p, end_p)
    hybrid_ret = calculate_period_return(hybrid_weights, start_p, end_p)  #C

    simulation_log.append({
        'date': decision_date, 'bh_return': bh_ret, 'hybrid_return': hybrid_ret,
        'weights': new_weights.copy(), 'reasoning': reasoning,
        'conviction': conviction, 'flags': flags
    })
    hybrid_weights = new_weights.copy()  #D

#A Buy-and-hold baseline maintains constant 20% weights throughout
#B Graceful degradation: if parsing fails, maintain current allocation
#C Returns calculated on weights held at period START, not new recommendations
#D Update weights for next period after calculating current period returns

## 12.7 Results and Interpretation
### Listing 12.10 — Performance calculation

In [ ]:
import matplotlib.pyplot as plt

results_df = pd.DataFrame(simulation_log)
results_df['bh_cum'] = (1 + results_df['bh_return']/100).cumprod() - 1  #A
results_df['hybrid_cum'] = (1 + results_df['hybrid_return']/100).cumprod() - 1

bh_total = results_df['bh_cum'].iloc[-1] * 100
hybrid_total = results_df['hybrid_cum'].iloc[-1] * 100

def calc_max_drawdown(returns):
    cum = (1 + returns/100).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min() * 100  #B

bh_mdd = calc_max_drawdown(results_df['bh_return'])
hybrid_mdd = calc_max_drawdown(results_df['hybrid_return'])
bh_vol = results_df['bh_return'].std()
hybrid_vol = results_df['hybrid_return'].std()

print(f"Buy-and-Hold  => Total: {bh_total:.2f}%, MaxDD: {bh_mdd:.2f}%, Vol: {bh_vol:.2f}%")
print(f"Hybrid (LLM)  => Total: {hybrid_total:.2f}%, MaxDD: {hybrid_mdd:.2f}%, Vol: {hybrid_vol:.2f}%")

#A Compound returns to get cumulative performance
#B Maximum drawdown: largest peak-to-trough decline during the period

### Listing 12.11 — Performance comparison panels
### Listing 12.12 — Diagnostic panels

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cumulative Returns
ax1 = axes[0, 0]
ax1.plot(results_df['date'], results_df['bh_cum']*100, 'b-o', label='Buy-and-Hold (20% each)', linewidth=2)
ax1.plot(results_df['date'], results_df['hybrid_cum']*100, 'r-o', label='Hybrid (LLM-Guided)', linewidth=2)
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.set_title('Cumulative Returns', fontweight='bold')
ax1.set_ylabel('Return (%)')
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.tick_params(axis='x', rotation=45)

# Monthly Returns
ax2 = axes[0, 1]
x = np.arange(len(results_df)); width = 0.35
ax2.bar(x - width/2, results_df['bh_return'], width, label='Buy-and-Hold', color='blue', alpha=0.7)
ax2.bar(x + width/2, results_df['hybrid_return'], width, label='Hybrid', color='red', alpha=0.7)
ax2.axhline(0, color='gray', linestyle='--')
ax2.set_title('Monthly Returns', fontweight='bold'); ax2.set_ylabel('Return (%)')
ax2.set_xticks(x); ax2.set_xticklabels([d[5:7] for d in results_df['date']])
ax2.set_xlabel('Month (2025)'); ax2.legend(); ax2.grid(True, alpha=0.3)

# Weight Evolution
ax3 = axes[1, 0]
weight_history = pd.DataFrame([r['weights'] for r in simulation_log])
weight_history.index = results_df['date']
weight_history.plot(kind='bar', stacked=True, ax=ax3, colormap='tab10')
ax3.set_title('Hybrid Portfolio Weight Evolution', fontweight='bold')
ax3.set_ylabel('Weight (%)'); ax3.set_xlabel('Decision Date')
ax3.legend(loc='upper right', fontsize=8); ax3.tick_params(axis='x', rotation=45)

# Summary Panel
ax4 = axes[1, 1]; ax4.axis('off')
summary = f"""
BACKTEST SUMMARY
{'='*45}
Period: {REBALANCE_DATES[0]} to {REBALANCE_DATES[-1]}
Assets: SPY, QQQ, VIXY, TLT, WEAT

               Buy-and-Hold   Hybrid (LLM)
               ------------   ------------
Total Return:  {bh_total:>10.2f}%   {hybrid_total:>10.2f}%
Max Drawdown:  {bh_mdd:>10.2f}%   {hybrid_mdd:>10.2f}%
Monthly Vol:   {bh_vol:>10.2f}%   {hybrid_vol:>10.2f}%
{'='*45}
Outperformance: {hybrid_total - bh_total:>10.2f}%
"""
ax4.text(0.05, 0.95, summary, fontsize=10, fontfamily='monospace', verticalalignment='top', transform=ax4.transAxes)
plt.tight_layout(); plt.savefig('ch12_backtest_results.png', dpi=150, bbox_inches='tight'); plt.show()

### Listing 12.13 — Decision log for human review

In [ ]:
for log in simulation_log:
    print(f"\n{log['date']} (Conviction: {log['conviction']}/10)")
    print(f"  Reasoning: {log['reasoning']}")
    w = log['weights']
    weight_str = " ".join([f"{etf}={w[etf]:.0f}%" for etf in ETF_LIST])  #A
    print(f"  Weights: {weight_str}")
    if log['flags']:
        print(f"  Flags: {log['flags']}")  #B

#A Compact weight display showing all five ETF allocations
#B Surface any flags raised by the LLM for human review